# 📊 QLoRA Fine-Tuning Project — Notebook 1: Setup & Data Pipeline
**Project:** Comparing Prompt Engineering vs QLoRA Fine-Tuning for Text Summarization  
**Dataset:** CNN/DailyMail (News Summarization Benchmark)  
**Platform:** Kaggle T4 GPU  

## What This Notebook Does
1. Installs all dependencies
2. Loads CNN/DailyMail dataset from HuggingFace
3. Explores token length distributions
4. Preprocesses and formats data (input/output pairs)
5. Creates train/val/test splits (8000/1000/1000)
6. Saves processed dataset for use in all subsequent notebooks

**Runtime:** ~15 minutes  
**Output:** Processed dataset saved to `/kaggle/working/qlora_project/`

In [ ]:
# ══════════════════════════════════════════════════════
# 📦 CELL 1: Install Dependencies
# ══════════════════════════════════════════════════════
# NOTE: Do NOT install numpy/pandas — Kaggle has compatible versions pre-installed
# Only install ML packages not included by default

!pip install -q transformers==4.40.0
!pip install -q datasets==2.18.0
!pip install -q accelerate==0.28.0
!pip install -q peft==0.10.0
!pip install -q bitsandbytes==0.43.1
!pip install -q evaluate==0.4.1
!pip install -q rouge-score==0.1.2
!pip install -q sentencepiece==0.2.0

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # suppress tokenizer warnings

print("✅ All ML dependencies installed")
print("   numpy/pandas/matplotlib/seaborn: using Kaggle pre-installed versions")

In [ ]:
# ══════════════════════════════════════════════════════
# ⚙️ CELL 2: Configuration
# ══════════════════════════════════════════════════════
import os, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

class Config:
    # Dataset
    DATASET_NAME    = "cnn_dailymail"
    DATASET_CONFIG  = "3.0.0"
    INPUT_FIELD     = "article"
    SUMMARY_FIELD   = "highlights"
    
    # Sizes
    TRAIN_SAMPLES   = 8000
    VAL_SAMPLES     = 1000
    TEST_SAMPLES    = 1000
    
    # Token limits
    MAX_INPUT_TOKENS  = 512   # consistent across all notebooks
    MAX_OUTPUT_TOKENS = 128
    MAX_DOC_TOKENS    = 1024  # filter out very long docs
    
    # Paths
    BASE_PATH         = "/kaggle/working/qlora_project"
    DATA_DIR          = f"{BASE_PATH}/data"
    PROCESSED_DIR     = f"{BASE_PATH}/processed_dataset"
    RESULTS_DIR       = f"{BASE_PATH}/results"

# Create directories
for d in [Config.DATA_DIR, Config.PROCESSED_DIR, Config.RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Config set")
print(f"   Train samples : {Config.TRAIN_SAMPLES:,}")
print(f"   Val samples   : {Config.VAL_SAMPLES:,}")
print(f"   Test samples  : {Config.TEST_SAMPLES:,}")
print(f"   Max input tok : {Config.MAX_INPUT_TOKENS}")
print(f"   Max output tok: {Config.MAX_OUTPUT_TOKENS}")

## 📥 Load Dataset

In [ ]:
# ══════════════════════════════════════════════════════
# 📥 CELL 3: Load CNN/DailyMail Dataset
# ══════════════════════════════════════════════════════
from datasets import load_dataset

print(f"📥 Loading {Config.DATASET_NAME}...")
raw_dataset = load_dataset(Config.DATASET_NAME, Config.DATASET_CONFIG)

print("\n✅ Dataset loaded!")
for split, data in raw_dataset.items():
    print(f"   {split:12s}: {len(data):,} examples")

# Show sample
sample = raw_dataset['train'][0]
print(f"\n🔍 Keys: {list(sample.keys())}")
print(f"\n📝 Sample article (first 300 chars):")
print(sample[Config.INPUT_FIELD][:300] + "...")
print(f"\n📝 Sample highlights:")
print(sample[Config.SUMMARY_FIELD])

## 🔤 Load Tokenizer & Analyze Token Distributions

In [ ]:
# ══════════════════════════════════════════════════════
# 🔤 CELL 4: Load Tokenizer
# ══════════════════════════════════════════════════════
from transformers import AutoTokenizer

MODEL_ID = "mistralai/Mistral-7B-v0.1"
print(f"Loading tokenizer: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

print(f"✅ Tokenizer loaded")
print(f"   Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
# ══════════════════════════════════════════════════════
# 📊 CELL 5: Analyze Token Length Distributions
# ══════════════════════════════════════════════════════
from tqdm import tqdm

print("📊 Analyzing token lengths on 2000 samples...")
ANALYSIS_SIZE = 2000
indices = np.random.choice(len(raw_dataset['train']), ANALYSIS_SIZE, replace=False)

input_lengths, output_lengths = [], []
for idx in tqdm(indices, desc="Tokenizing"):
    sample = raw_dataset['train'][int(idx)]
    input_lengths.append(len(tokenizer(sample[Config.INPUT_FIELD], add_special_tokens=False)['input_ids']))
    output_lengths.append(len(tokenizer(sample[Config.SUMMARY_FIELD], add_special_tokens=False)['input_ids']))

input_lengths  = np.array(input_lengths)
output_lengths = np.array(output_lengths)

print(f"\n📈 Input (Article) Token Stats:")
print(f"   Mean  : {input_lengths.mean():.0f}")
print(f"   Median: {np.median(input_lengths):.0f}")
print(f"   P90   : {np.percentile(input_lengths, 90):.0f}")
print(f"   P95   : {np.percentile(input_lengths, 95):.0f}")
print(f"   Max   : {input_lengths.max():.0f}")

print(f"\n📈 Output (Highlights) Token Stats:")
print(f"   Mean  : {output_lengths.mean():.0f}")
print(f"   Median: {np.median(output_lengths):.0f}")
print(f"   P90   : {np.percentile(output_lengths, 90):.0f}")
print(f"   P95   : {np.percentile(output_lengths, 95):.0f}")
print(f"   Max   : {output_lengths.max():.0f}")

In [ ]:
# ══════════════════════════════════════════════════════
# 📊 CELL 6: Plot & Save Token Distribution
# ══════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CNN/DailyMail Token Length Distributions", fontsize=14, fontweight='bold')

# Input lengths
axes[0].hist(input_lengths, bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(Config.MAX_INPUT_TOKENS, color='red', linestyle='--', linewidth=2, label=f'Truncation ({Config.MAX_INPUT_TOKENS})')
axes[0].set_xlabel("Token Length")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Article (Input) Length Distribution")
axes[0].legend()

# Output lengths
axes[1].hist(output_lengths, bins=50, color='coral', alpha=0.8, edgecolor='white')
axes[1].axvline(Config.MAX_OUTPUT_TOKENS, color='red', linestyle='--', linewidth=2, label=f'Limit ({Config.MAX_OUTPUT_TOKENS})')
axes[1].set_xlabel("Token Length")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Highlights (Output) Length Distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{Config.RESULTS_DIR}/token_distributions.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved token_distributions.png")

## 🔧 Preprocess & Split Dataset

In [ ]:
# ══════════════════════════════════════════════════════
# 🔧 CELL 7: Preprocess Function
# ══════════════════════════════════════════════════════
def preprocess_function(examples):
    inputs, outputs, keep_mask = [], [], []
    
    for article, highlights in zip(examples[Config.INPUT_FIELD], examples[Config.SUMMARY_FIELD]):
        if not article or not highlights:
            keep_mask.append(False)
            inputs.append("")
            outputs.append("")
            continue
        
        # Tokenize to check length
        input_tokens = tokenizer(article, add_special_tokens=False)['input_ids']
        
        # Filter extremely long docs
        if len(input_tokens) > Config.MAX_DOC_TOKENS:
            keep_mask.append(False)
            inputs.append("")
            outputs.append("")
            continue
        
        # Truncate input
        if len(input_tokens) > Config.MAX_INPUT_TOKENS:
            input_tokens = input_tokens[:Config.MAX_INPUT_TOKENS]
            article = tokenizer.decode(input_tokens, skip_special_tokens=True)
        
        # Truncate output
        output_tokens = tokenizer(highlights, add_special_tokens=False)['input_ids']
        if len(output_tokens) > Config.MAX_OUTPUT_TOKENS:
            output_tokens = output_tokens[:Config.MAX_OUTPUT_TOKENS]
            highlights = tokenizer.decode(output_tokens, skip_special_tokens=True)
        
        inputs.append(article.strip())
        outputs.append(highlights.strip())
        keep_mask.append(True)
    
    return {"input": inputs, "output": outputs, "keep": keep_mask}

print("✅ Preprocessing function defined")

In [ ]:
# ══════════════════════════════════════════════════════
# ⚙️ CELL 8: Apply Preprocessing
# ══════════════════════════════════════════════════════
from datasets import DatasetDict

print("🔄 Preprocessing dataset...")
processed = {}

for split in ['train', 'validation', 'test']:
    print(f"   Processing {split}...")
    result = raw_dataset[split].map(
        preprocess_function,
        batched=True,
        batch_size=500,
        remove_columns=raw_dataset[split].column_names,
        desc=f"Processing {split}"
    )
    # Filter out removed examples
    result = result.filter(lambda x: x['keep'])
    result = result.remove_columns(['keep'])
    processed[split] = result
    print(f"   {split}: {len(result):,} examples after filtering")

print("\n✅ Preprocessing complete")

In [ ]:
# ══════════════════════════════════════════════════════
# ✂️ CELL 9: Subsample to Target Sizes
# ══════════════════════════════════════════════════════
np.random.seed(42)
final_splits = {}

for split, target_size in [
    ('train',      Config.TRAIN_SAMPLES),
    ('validation', Config.VAL_SAMPLES),
    ('test',       Config.TEST_SAMPLES)
]:
    current = processed[split]
    if len(current) > target_size:
        indices = np.random.choice(len(current), target_size, replace=False)
        final_splits[split] = current.select(indices)
        print(f"   {split:12s}: {len(current):,} → {target_size:,} (subsampled)")
    else:
        final_splits[split] = current
        print(f"   {split:12s}: {len(current):,} (kept all)")

print("\n✅ Final dataset sizes:")
for split, data in final_splits.items():
    print(f"   {split:12s}: {len(data):,}")

## 💾 Save Dataset

In [ ]:
# ══════════════════════════════════════════════════════
# 💾 CELL 10: Save as HuggingFace Dataset
# ══════════════════════════════════════════════════════
from datasets import DatasetDict

final_dataset = DatasetDict(final_splits)
final_dataset.save_to_disk(Config.PROCESSED_DIR)
print(f"✅ Saved HuggingFace dataset to {Config.PROCESSED_DIR}")

# Also save as JSON for easy inspection
for split, data in final_splits.items():
    json_path = os.path.join(Config.DATA_DIR, f"{split}.json")
    records = [{"input": ex["input"], "output": ex["output"]} for ex in data]
    with open(json_path, "w") as f:
        json.dump(records, f, indent=2)
    print(f"   Saved {split}.json ({len(records):,} records)")

In [ ]:
# ══════════════════════════════════════════════════════
# 🔍 CELL 11: Sanity Check & Sample Preview
# ══════════════════════════════════════════════════════
print("🔍 Sample Examples from Processed Dataset:\n")
print("=" * 80)

for i in range(2):
    ex = final_splits['train'][i]
    print(f"\n📌 Example {i+1}")
    print(f"INPUT  ({len(ex['input'].split())} words):")
    print(ex['input'][:400] + "...")
    print(f"\nOUTPUT ({len(ex['output'].split())} words):")
    print(ex['output'])
    print("─" * 80)

print("\n✅ Dataset looks good!")

In [ ]:
# ══════════════════════════════════════════════════════
# 📊 CELL 12: Save Config & Dataset Stats
# ══════════════════════════════════════════════════════
stats = {
    "dataset": Config.DATASET_NAME,
    "model_id": "mistralai/Mistral-7B-v0.1",
    "splits": {
        split: {
            "size": len(data),
            "avg_input_words": float(np.mean([len(ex["input"].split()) for ex in data])),
            "avg_output_words": float(np.mean([len(ex["output"].split()) for ex in data]))
        }
        for split, data in final_splits.items()
    },
    "config": {
        "max_input_tokens":  Config.MAX_INPUT_TOKENS,
        "max_output_tokens": Config.MAX_OUTPUT_TOKENS,
        "max_doc_tokens":    Config.MAX_DOC_TOKENS,
        "train_samples":     Config.TRAIN_SAMPLES,
        "val_samples":       Config.VAL_SAMPLES,
        "test_samples":      Config.TEST_SAMPLES,
    }
}

stats_path = os.path.join(Config.RESULTS_DIR, "dataset_stats.json")
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)

print("📊 Dataset Statistics:")
print(json.dumps(stats, indent=2))
print(f"\n✅ Saved to {stats_path}")

## ✅ Notebook 1 Complete!

### What was saved:
| File | Location | Description |
|------|----------|-------------|
| `processed_dataset/` | `/kaggle/working/qlora_project/` | HuggingFace dataset format |
| `train.json` | `data/` | 8,000 training examples |
| `validation.json` | `data/` | 1,000 validation examples |
| `test.json` | `data/` | 1,000 test examples |
| `token_distributions.png` | `results/` | Token length analysis plot |
| `dataset_stats.json` | `results/` | Dataset statistics |